In this notebook, I will build embedding for ICA paper title + abstracts. 

In [116]:
import pandas as pd 
import numpy as np 
from tqdm.auto import tqdm
from dotenv import load_dotenv
load_dotenv()
import os

In [117]:
df = pd.read_csv('processed/papers.csv')
df.shape

(27466, 9)

In [118]:
df.columns

Index(['Paper ID', 'Title', 'Paper Type', 'Abstract', 'Number of Authors',
       'Year', 'Session', 'Division/Unit', 'Authors'],
      dtype='object')

In [119]:
title_missing = (
    df["Title"].isna() |
    (df["Title"].astype(str).str.strip() == "")
).sum()

abstract_missing = (
    df["Abstract"].isna() |
    (df["Abstract"].astype(str).str.strip() == "")
).sum()

print(f"Title missing: {title_missing}")
print(f"Abstract missing: {abstract_missing}")

Title missing: 0
Abstract missing: 23


In [120]:
# Add title + abstract
df["embedding_text"] = (
    df["Title"].fillna("").str.strip()
    + "\n\n"
    + df["Abstract"].fillna("").str.strip()
)

In [121]:
paper_ids = df['Paper ID'].tolist()
paper_titles = df.Title.tolist()
paper_abstracts = df.Abstract.tolist()
# Create dictionaries
Id2Title = dict(zip(paper_ids, paper_titles))
Id2Abstract = dict(zip(paper_ids, paper_abstracts))

In [122]:
# from sentence_transformers import SentenceTransformer
# from sklearn.preprocessing import normalize
# model = SentenceTransformer('all-MiniLM-L6-v2')
# from typing import List, Dict, Tuple
# import faiss 

In [123]:
# def get_paper_embeddings(
#     paper_ids: List[str],
#     paper_texts: List[str],
#     model: SentenceTransformer,
#     batch_size: int = 32,
# ) -> Dict[str, np.ndarray]:

#     if len(paper_ids) != len(paper_texts):
#         raise ValueError(
#             "paper_ids and paper_texts must have the same length."
#         )

#     embeddings = model.encode(
#         paper_texts,
#         batch_size=batch_size,
#         show_progress_bar=True,
#         normalize_embeddings=True,
#         convert_to_numpy=True,
#     )

#     return dict(zip(paper_ids, embeddings))

In [124]:
# paper_embeddings = get_paper_embeddings(
#     paper_ids,
#     df["embedding_text"].tolist(),
#     model,
# )

In [125]:
# def prepare_faiss_index(
#     paper_embeddings: Dict[str, np.ndarray],
# ) -> Tuple[faiss.Index, List[str]]:
#     """
#     Prepare a FAISS index from paper embeddings.

#     Args:
#         paper_embeddings: Dictionary mapping paper IDs to normalized embeddings.

#     Returns:
#         Tuple of (FAISS index, ordered list of paper IDs).
#     """
#     paper_ids = list(paper_embeddings.keys())

#     embeddings = np.array(
#         [paper_embeddings[pid] for pid in paper_ids]
#     ).astype("float32")

#     dim = embeddings.shape[1]

#     index = faiss.IndexFlatIP(dim)
#     index.add(embeddings)

#     return index, paper_ids

In [126]:
# def search_papers(
#     faiss_index: faiss.Index,
#     paper_ids: List[str],
#     id2title_dict: Dict[str, str],
#     id2abstract_dict: Dict[str, str],
#     user_query: str,
#     model: SentenceTransformer,
#     k: int = 10,
# ) -> pd.DataFrame:
#     """
#     Search papers based on a user query using FAISS and cosine similarity.

#     Args:
#         faiss_index: Pre-built FAISS index.
#         paper_ids: Ordered list of paper IDs corresponding to the FAISS index.
#         id2title_dict: Dict mapping paper IDs to paper titles.
#         id2abstract_dict: Dict mapping paper IDs to paper abstracts.
#         user_query: User search query.
#         model: SentenceTransformer model.
#         k: Number of results to return.

#     Returns:
#         DataFrame with top-k paper IDs, titles, abstracts, and similarity scores.
#     """
#     query_embedding = model.encode(
#         [user_query],
#         normalize_embeddings=True,
#         convert_to_numpy=True,
#     ).astype("float32")

#     similarities, indices = faiss_index.search(query_embedding, k)

#     results = []

#     for idx, score in zip(indices[0], similarities[0]):
#         paper_id = paper_ids[idx]

#         results.append({
#             "Paper ID": paper_id,
#             "Title": id2title_dict.get(paper_id, ""),
#             "Abstract": id2abstract_dict.get(paper_id, ""),
#             "Score": float(score),
#         })

#     return pd.DataFrame(results)

In [127]:
# faiss_index, ordered_paper_ids = prepare_faiss_index(paper_embeddings)

In [128]:
# user_query = "social media misinformation and political polarization"

# results = search_papers(
#     faiss_index=faiss_index,
#     paper_ids=ordered_paper_ids,
#     id2title_dict=Id2Title,
#     id2abstract_dict=Id2Abstract,
#     user_query=user_query,
#     model=model,
#     k=10,
# )

# results

In [129]:
from openai import OpenAI
import numpy as np

# OpenAI client
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Where is France?"
)

embedding = np.array(response.data[0].embedding)

print(f"text-embedding-3-small's output has a dimension of {len(embedding)}")

norm = np.linalg.norm(embedding)

print(f"L2 norm：{norm}")

if np.isclose(norm, 1, atol=1e-3):
    print("The vector is normalized.")
else:
    print("The vector is NOT normalized.")

text-embedding-3-small 的 output 维度为 1536
L2 范数：1.000190109201513
向量已进行 L2 归一化


In [130]:
from pinecone import Pinecone, ServerlessSpec
from concurrent.futures import ThreadPoolExecutor
import time
from typing import List

# Pinecone client
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Index name
index_name = "ica-conf-papers"

In [131]:
def get_openai_embeddings(texts: List[str]) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    embeddings = np.array(
        [item.embedding for item in response.data],
        dtype=np.float32
    )

    # normalize again for safety
    embeddings = embeddings / np.linalg.norm(
        embeddings,
        axis=1,
        keepdims=True
    )

    return embeddings

In [132]:
# Delete existing index if it exists
existing_indexes = [index["name"] for index in pc.list_indexes()]

if index_name in existing_indexes:
    print(f"Deleting existing index: {index_name}")
    pc.delete_index(index_name)
    time.sleep(10)
else:
    print(f"Index '{index_name}' does not exist.")

Deleting existing index: ica-conf-papers


In [133]:
# Check vector dimension
test_embedding = get_openai_embeddings([df["embedding_text"].iloc[0]])
vector_size = test_embedding.shape[1]

print(f"Vector dimension：{vector_size}")

向量维度为：1536


In [134]:
# Create Pinecone index
pc.create_index(
    name=index_name,
    dimension=vector_size,   # should be 1536
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

time.sleep(10)

In [135]:
index = pc.Index(index_name)

In [136]:
def clean_metadata_value(x):
    if pd.isna(x):
        return ""
    return str(x)

In [137]:
def process_one_batch(
    batch_ids: List,
    batch_texts: List[str],
    index,
):
    try:
        batch_embeddings = get_openai_embeddings(batch_texts)

        batch_vectors = []

        for j, paper_id in enumerate(batch_ids):
            batch_vectors.append({
                "id": str(paper_id),
                "values": batch_embeddings[j].tolist(),
                "metadata": {
                    "title": clean_metadata_value(Id2Title.get(paper_id, "")),
                    "abstract": clean_metadata_value(Id2Abstract.get(paper_id, ""))
                }
            })

        index.upsert(vectors=batch_vectors)

        return True

    except Exception as e:
        print(f"Error uploading batch: {e}")
        return False

In [138]:
paper_texts = df["embedding_text"].tolist()

batch_size = 32
all_batches = []

for i in range(0, len(paper_ids), batch_size):
    batch_ids = paper_ids[i:i + batch_size]
    batch_texts = paper_texts[i:i + batch_size]
    all_batches.append((batch_ids, batch_texts))

In [140]:
# max_workers = 1

# with ThreadPoolExecutor(max_workers=max_workers) as executor:
#     results = list(tqdm(
#         executor.map(
#             lambda batch: process_one_batch(
#                 batch[0],
#                 batch[1],
#                 index
#             ),
#             all_batches
#         ),
#         total=len(all_batches),
#         desc="Uploading paper batches"
#     ))

# successful_batches = results.count(True)

# print(f"成功处理 {successful_batches}/{len(all_batches)} 批次")

# # Done in 20m

In [141]:
def search_pinecone(
    index,
    query_text: str,
    top_k: int = 5,
):
    query_embedding = get_openai_embeddings(
        [query_text]
    )[0]

    results = index.query(
        vector=query_embedding.tolist(),
        top_k=top_k,
        include_metadata=True
    )

    papers = []

    for match in results["matches"]:
        papers.append({
            "paper_id": match["id"],
            "title": match["metadata"].get("title", ""),
            "abstract": match["metadata"].get("abstract", ""),
            "similarity_score": match["score"]
        })

    return papers

In [142]:
results = search_pinecone(
    index=index,
    query_text="social media misinformation and political polarization",
    top_k=10
)

pd.DataFrame(results)

,paper_id,title,abstract,similarity_score
0,2017-a8a341f2560955e0,Lifecycle analysis of political misinformation...,This study examines how political misinformati...,0.653617
1,2018-4d655d18ae697587,"Misinformation, Social Media Habits, and News ...",Amid concerns about the spread of misinformati...,0.624174
2,2017-59bb0381c591e4c7,From Partisan Media to Misperception: Affectiv...,The prevalence and persistence of political mi...,0.606355
3,2010-99e15841f341da19,The Troubling Consequences of Online Election ...,A national telephone survey reveals that polit...,0.603950
4,2013-6b86529da910d4a5,Political Polarization as a Function of Citize...,Pundits and other observers of contemporary de...,0.602557
5,2018-3eea95ab85eef0ba,Does Mainstream Tabloid News Poison the Well o...,News producers’ and citizens’ routine use of s...,0.598456
6,2018-a5635079d9f36a7c,The Affective Roots of Political Disagreement ...,Political disagreement is more prevalent on so...,0.586258
7,2018-21cb256d1652ab8f,Social Media Effects on Opinion Polarization a...,The selective exposure to likeminded political...,0.584596
8,2018-17a7b3b4e63edd4f,The Spread of Self-Generated Political Misinfo...,How is misinformation created and how does it ...,0.580041
9,2017-6ffbf7968ede5083,“Why Not Give Them a Chance?”: Information See...,Relying on in-depth interviews (n =37) with po...,0.578143
